In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-12-11 20:50:08 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Hallazgos

Tabla `resultados_wompi.wompi_transactions`

- La tabla se ingesta incrementalmente. 
- Se debe adicionar el filtro `estado_transaccion = 'Aprobada'`
- El monto_transaccion está en centavos, se debe dividir en 100 para transformarlo en pesos
- La variable que representa la fecha de transaccion es `fecha_creacion_transaccion`
- El detalle de la fecha de la transacción es hasta minutos
- La información se ingesta t menos 1 día [sucede en octubre y noviembre de 2025, seguramente en los siguientes meses], esto se mantiene desde el 2023. Sin embargo, se observa que para esos mismos meses del año 2022 la ingestión de las compras diarias se realiza el último día de cada mes.


## Análisis ingestión compras tabla transaccional wompi

In [8]:
sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          fecha_creacion_transaccion,
          count(*) AS num_compras
   FROM resultados_wompi.wompi_transactions
   WHERE YEAR = 2025
     AND MONTH BETWEEN 11 AND 12
     AND DAY BETWEEN 1 AND 31
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY fecha_creacion_transaccion DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY fecha_creacion_transaccion) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY fecha_creacion_transaccion
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY fecha_creacion_transaccion DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY fecha_creacion_transaccion
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY fecha_creacion_transaccion DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
df_ingestiones = helper.obtener_dataframe(sql)

2025-12-11 23:19:06 - [INFO] - Transcurrido: 7561, Tiempo de Refresco = 1000


---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 37/37 DATAFRAME                                       descargando   11:19:06 PM             

2025-12-11 23:19:08 - [INFO] - 40 filas, 10 columnas, 00:01.7 consultando, 00:00.2 descargando, 00:00.0 convirtiendo


 37/37 DATAFRAME                                        finalizado   11:19:06 PM     00:02.1 
---------------------------------------------------------------------------------------------


In [9]:
df_ingestiones

,year,month,day,fecha_creacion_transaccion,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
0,2025,12,10,20251210,504259,504259,504259,1,1.0,1.0
1,2025,12,9,20251209,517270,517270,517270,1,1.0,1.0
2,2025,12,8,20251208,302096,302096,302096,1,1.0,1.0
3,2025,12,7,20251207,307138,307138,307138,1,1.0,1.0
4,2025,12,6,20251206,359941,359941,359941,1,1.0,1.0
5,2025,12,5,20251205,511922,511922,511922,1,1.0,1.0
6,2025,12,4,20251204,469341,469341,469341,1,1.0,1.0
7,2025,12,3,20251203,481813,481813,481813,1,1.0,1.0
8,2025,12,2,20251202,554149,554149,554149,1,1.0,1.0
9,2025,12,1,20251201,638813,638813,638813,1,1.0,1.0


## Construcción histórico transacciones

In [ ]:
# # Construcción histórico trxs
# # Se selecciona el rango de tiempo de las ingestiones a almacenar [Esto para efectos de facilitar la actualización del histórico]
# fecha_inicial = '2025-11-15' # MODIFICAR. DEBE SER EL PRIMER DÍA DE INGESTIÓN DE TRANSACCIONES A ALMACENAR O EL SIGUIENTE DÍA DESPUÉS DEL ÚLTIMO EN UNA ACTUALIZACIÓN.
# fecha_final = '2025-11-30' # MODIFICAR. DEBE SER EL ÚLTIMO DÍA DE INGESTIÓN DE TRANSACCIONES ALMACENADAS O EL DÍA MÁS RECIENTE EN UNA ACTUALIZACIÓN
# fecha_inicial_ts = pd.to_datetime(fecha_inicial)
# fecha_final_ts = pd.to_datetime(fecha_final)
# fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='D')

# # # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# # pri_dia_part = fechas[-1] + relativedelta(days=1)
# # pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# # ult_dia_part = fechas[-1] + relativedelta(days=10)
# # ult_dia_part = ult_dia_part.date().isoformat()
# # fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# # fechas = fechas.append(fechas_faltantes)

# # df_config
# df_config = pd.DataFrame({'fechas': fechas})
# df_config['year'] = df_config['fechas'].dt.year
# df_config['month'] = df_config['fechas'].dt.month
# df_config['day'] = df_config['fechas'].dt.day
# # df_config['fechas_fin_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1) - relativedelta(days=1))
# # df_config['year_fin_mes'] = df_config['fechas_fin_mes'].dt.year
# # df_config['month_fin_mes'] = df_config['fechas_fin_mes'].dt.month
# # df_config['day_fin_mes'] = df_config['fechas_fin_mes'].dt.day
# df_config

,fechas,year,month,day
0,2025-11-15,2025,11,15
1,2025-11-16,2025,11,16
2,2025-11-17,2025,11,17
3,2025-11-18,2025,11,18
4,2025-11-19,2025,11,19
5,2025-11-20,2025,11,20
6,2025-11-21,2025,11,21
7,2025-11-22,2025,11,22
8,2025-11-23,2025,11,23
9,2025-11-24,2025,11,24


In [ ]:
# Crear tabla que almacenará la información
# EN ACTUALIZACIÓN SE VULEVE A CREAR LA TABLA DESDE CERO
drop_sql = """DROP TABLE IF EXISTS proceso_vdm.mdo_wompi_trxs PURGE;"""
helper.ejecutar_consulta(drop_sql)

sql = """
CREATE TABLE proceso_vdm.mdo_wompi_trxs  (
                cod_unico VARCHAR,
                f_trx INT,
                num_trxs BIGINT,
                mnt_total_trxs DECIMAL(38,2)
                )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_trxs;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 38/38      DROP           proceso_vdm.mdo_wompi_trxs   finalizado   11:24:32 PM     00:00.5 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 39/39    CREATE           proceso_vdm.mdo_wompi_trxs   finalizado   11:24:33 PM     00:00.2 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

In [13]:
# OJO: ESTA QUEMADO EL ÚLTIMO AÑO DE INGESTIÓN. SE QUEMO EL 2030

print('#' * 50)
print('')
print('Obteniendo transacciones wompi de los comercios')
print('')

sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_trxs_temp PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_wompi_trxs_temp STORED AS PARQUET AS
SELECT cast(id_comercio as varchar) as cod_unico,
    fecha_creacion_transaccion as f_trx,
    count(*) AS num_trxs,
    CAST(sum(monto_transaccion)/100 AS DECIMAL(38,2)) AS mnt_total_trxs
FROM resultados_wompi.wompi_transactions
WHERE LOWER(TRIM(estado_transaccion)) = "aprobada"
AND YEAR BETWEEN 2022 AND 2030
AND MONTH BETWEEN 1 AND 12
AND DAY BETWEEN 1 AND 31
GROUP BY 1,
            2;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_trxs_temp;"""
helper.ejecutar_consulta(sql_compute)

print('')
print('Insertando transacciones wompi de los comercios')
print('')

sql = """
INSERT INTO proceso_vdm.mdo_wompi_trxs
SELECT cod_unico,
        f_trx,
        num_trxs,
        mnt_total_trxs
FROM proceso.mdo_wompi_trxs_temp;"""
helper.ejecutar_consulta(sql)
print('')

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_trxs;"""
helper.ejecutar_consulta(sql_compute)

##################################################

Obteniendo transacciones wompi de los comercios

---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 46/46      DROP          proceso.mdo_wompi_trxs_temp   finalizado   11:26:05 PM     00:00.3 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 47/47    CREATE          proceso.mdo_wompi_trxs_temp   finalizado   11:26:06 PM     00:02.9 
-----------------------------------------------------

## Construcción histórico transacciones por mes

In [14]:
# Crear tabla que almacenará las transacciones menusales por cliente
# EN ACTUALIZACIÓN SE VULEVE A CREAR LA TABLA DESDE CERO
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_wompi_trxs_mes_1 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_wompi_trxs_mes_1 (
                cod_unico VARCHAR,
                num_trxs BIGINT,
                mnt_total_trxs DECIMAL(38,2),
                periodo_trxs INT
            )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_trxs_mes_1;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 51/51      DROP     proceso_vdm.mdo_wompi_trxs_mes_1   finalizado   11:26:27 PM     00:00.2 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 52/52    CREATE     proceso_vdm.mdo_wompi_trxs_mes_1   finalizado   11:26:27 PM     00:00.2 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

In [16]:
print('#' * 50)
print('')
print('Obteniendo transacciones wompi de los comercios')
print('')

sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_trxs_mes_1_temp PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_wompi_trxs_mes_1_temp STORED AS PARQUET AS
SELECT cod_unico,
       cast(replace(left(cast(f_trx AS string), 6), '-', '') AS INT) AS periodo_trxs,
       count(*) AS num_trxs,
       sum(mnt_total_trxs) AS mnt_total_trxs
FROM proceso_vdm.mdo_wompi_trxs
GROUP BY 1,
       2;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_trxs_mes_1_temp;"""
helper.ejecutar_consulta(sql_compute)

print('')
print('Insertando transacciones wompi de los comercios')
print('')

sql = """
INSERT INTO proceso_vdm.mdo_wompi_trxs_mes_1
SELECT cod_unico,
       num_trxs,
       mnt_total_trxs,
       periodo_trxs
FROM proceso.mdo_wompi_trxs_mes_1_temp;"""
helper.ejecutar_consulta(sql)
print('')

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_trxs_mes_1;"""
helper.ejecutar_consulta(sql_compute)

##################################################

Obteniendo transacciones wompi de los comercios

---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 59/59      DROP    proceso.mdo_wompi_trxs_mes_1_temp   finalizado   11:26:49 PM     00:00.5 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 60/60    CREATE    proceso.mdo_wompi_trxs_mes_1_temp   finalizado   11:26:50 PM     00:03.1 
-----------------------------------------------------

## Construcción históricos

Se calcula la métrica uso para tres escenarios

- Todos los clientes
- Clientes nuevos
- Clientes viejos

Se dice que un cliente [todos, nuevos o viejos] tiene uso cuando realiza al menos una (1) transaccion en el año corriente. 

Importante tener en cuenta para clientes nuevos. Un cliente que vinculó adquirencia en el año 2024 [nuevo en 2024] y realizó un transacción en el mismo año suma a la métrica, en cambio si ese mismo cliente realiza una transacción en el año 2025 y siguiente no sumará a la métrica.

**Atención:** los históricos de wompi se almacenarán en la tabla `proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist` creada en el archivo *uso_adquirencia.ipynb*

In [ ]:
# # Tabla que almacenará comercios con trxs por periodo
# sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_wompi_vinculaciones_con_trxs_hist PURGE;"""
# helper.ejecutar_consulta(sql_drop)

# sql = """
# CREATE TABLE proceso_vdm.mdo_wompi_vinculaciones_con_trxs_hist  (
#                 codigo_unico DOUBLE,
#                 periodo DOUBLE
#                 )
# STORED AS PARQUET
# TBLPROPERTIES ('transactional' = 'false');
# """
# helper.ejecutar_consulta(sql)

# sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinculaciones_con_trxs_hist;"""
# helper.ejecutar_consulta(sql_compute)

-----------------------------------------------------------------------------------------------------
     i       tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------------
 1816/1816      DROP ...mdo_wompi_vinculaciones_con_trxs_hist   finalizado   03:47:27 AM     00:00.1 
-----------------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------------
     i       tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------------
 1817/1817    CREATE ...mdo_wompi_vinculaciones_con_trxs_hist   finalizado   03:47:27 AM     00:00.1 
----------------------------------------------------------------------------------

In [17]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2025-11-01' # MODIFICAR. INICIO DE UN MES. EN ACTUALIZACIÓN USAR EL SIGUIENTE MES DESPUÉS DEL ÚLTIMO ALMACENADO
fecha_final = '2025-11-30' # MODIFICAR. FIN DE UN MES. EN ACTUALIZACIÓN USAR EL MES RECIENTE CON TRANSACCIONES COMPLETAS
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='M')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['periodo'] = df_config['year'] * 100 + df_config['month']
df_config['periodo_base_year'] = df_config['year'] * 100 + 1
df_config['periodo_sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config['year_sgte_mes'] = df_config['periodo_sgte_mes'].dt.year
df_config['periodo_sgte_mes'] = df_config['periodo_sgte_mes'].apply(lambda x: x.year * 100 + x.month)
df_config['periodo_viejos'] = df_config['fechas'].apply(lambda x: str(x.year - 1) + '12')
# df_config['ult_6_meses_fin'] = df_config['fechas'].apply(lambda x: x - relativedelta(months=5))
# df_config['ult_6_meses_inicio'] = df_config['ult_6_meses_fin'].apply(lambda x: x.replace(day=1))
# df_config['year_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.year
# df_config['month_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.month
# df_config['periodo_ult_6_meses'] = df_config['year_ult_6_meses'] * 100 + df_config['month_ult_6_meses']
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_sgte_mes,year_sgte_mes,periodo_viejos
0,2025-11-30,2025,11,30,202511,202501,202512,2025,202412


In [18]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch

2025-12-11 23:27:39 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-12-11 23:27:40 - [INFO] - Finalizo la busqueda, duracion: 00:00.6, resultado: {'year': 2025, 'month': 12, 'day': 11}


{'year': 2025, 'month': 12, 'day': 11}

In [20]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Obteniendo Métrica 5X')
    print('')
    print('Mes de Análisis: ', str(row.year), '-', str(row.month))
    print('')
    print('Viejos')
    print('')
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
       AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
       AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
       AND modelo = 'Agregador'
       AND activo = 'A'
       AND desembolsos_permitidos = 'Si'
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc
    WHERE periodo <= """ + str(row.periodo_viejos) + """;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'viejos' as tipo_cliente,
         'wompi' as producto
    FROM proceso_vdm.mdo_wompi_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_wompi_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_wompi_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')


    print('Nuevos')
    print('')
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
       AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
       AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
       AND modelo = 'Agregador'
       AND activo = 'A'
       AND desembolsos_permitidos = 'Si'
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc
    WHERE periodo BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'nuevos' as tipo_cliente,
         'wompi' as producto
    FROM proceso_vdm.mdo_wompi_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_wompi_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_wompi_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')


    print('Todos')
    print('')
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
       AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
       AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
       AND modelo = 'Agregador'
       AND activo = 'A'
       AND desembolsos_permitidos = 'Si'
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc
    WHERE periodo <= """ + str(row.periodo) + """;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'todos' as tipo_cliente,
         'wompi' as producto
    FROM proceso_vdm.mdo_wompi_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_wompi_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_wompi_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')
    

##################################################

Obteniendo Métrica 5X

Mes de Análisis:  2025 - 11

Viejos

Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis

---------------------------------------------------------------------------------------------
   i     tipo                   nombre                   estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 64/64      DROP proceso.mdo_wompi_vinculaciones_temp   finalizado   11:28:11 PM     00:00.4 
---------------------------------------------------------------------------------------------

    CREATE TABLE proceso.mdo_wompi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = 2025
       AND MONTH = 12
       AND DAY = 11
       

In [22]:
# Obtener uso por mes
# Número de Vinculaciones
sql = """
SELECT producto,
       tipo_cliente,
       periodo,
       count(*) AS num_vinc_uso_cumsum_ym
FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
GROUP BY 1,
         2,
         3
ORDER BY periodo DESC, tipo_cliente, producto;
"""
df_prueba = helper.obtener_dataframe(sql)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 86/86 DATAFRAME                                           descargando   11:29:46 PM             

2025-12-11 23:29:49 - [INFO] - 282 filas, 4 columnas, 00:03.2 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 86/86 DATAFRAME                                            finalizado   11:29:46 PM     00:03.7 
-------------------------------------------------------------------------------------------------


In [23]:
df_prueba.head(20)

,producto,tipo_cliente,periodo,num_vinc_uso_cumsum_ym
0,adqui,nuevos,202511.0,23445
1,wompi,nuevos,202511.0,13808
2,adqui,todos,202511.0,102281
3,wompi,todos,202511.0,31058
4,adqui,viejos,202511.0,78836
5,wompi,viejos,202511.0,17250
6,adqui,nuevos,202510.0,21390
7,wompi,nuevos,202510.0,11275
8,adqui,todos,202510.0,99996
9,wompi,todos,202510.0,28226
